In [40]:
import pandas as pd
import os

In [42]:
raw_file = r"D:\data_proj\data\online+retail+ii\online_retail_II.xlsx"
print("1/5 Loading year 2009-2010")
df_2009 = pd.read_excel(raw_file, sheet_name="Year 2009-2010")

print("2/5 Loading year 2010-2011")
df_2010 = pd.read_excel(raw_file , sheet_name="Year 2010-2011")

df = pd.concat([df_2009, df_2010], ignore_index = True)
print(f"Total raw rows: {len(df):,}")


1/5 Loading year 2009-2010
2/5 Loading eyar 2010-2011
Total raw rows: 1,067,371


In [43]:
# Drop guest checkouts (no Customer ID)
#-> meaning churn cannot be analyzed since they did not login , anonymous users
df_clean = df.dropna(subset=['Customer ID']).copy()

In [44]:
# Remove cancellations ('C' in Invoice) and negative/zero quantities
#-> so we only analyze completed sales
df_clean['Invoice'] = df_clean['Invoice'].astype(str)
df_clean = df_clean[~df_clean['Invoice'].str.startswith('C')]
df_clean = df_clean[df_clean['Quantity'] > 0]

In [45]:
# Remove zero or negative prices (bad bookkeeping rows)
df_clean = df_clean[df_clean['Price'] > 0]

# Create TotalPrice column (Quantity * Unit Price)
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['Price']

In [46]:
# Fix data types
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

In [48]:
# 4. Save clean dataset to Parquet
print("4/5 Saving clean parquet file...")
os.makedirs(r"d:\data_proj\data\processed", exist_ok=True)

# Convert text columns so Parquet doesn't get confused by mixed types
df_clean['StockCode'] = df_clean['StockCode'].astype(str)
df_clean['Description'] = df_clean['Description'].astype(str)
df_clean['Country'] = df_clean['Country'].astype(str)
# Now save to parquet
os.makedirs(r"d:\data_proj\data\processed", exist_ok=True)
df_clean.to_parquet(r"d:\data_proj\data\processed\clean_retail.parquet", index=False)
print("SUCCESS! Saved clean dataset to data/processed/clean_retail.parquet")


df_clean.to_parquet(r"d:\data_proj\data\processed\clean_retail.parquet", index=False)

4/5 Saving clean parquet file...
SUCCESS! Saved clean dataset to data/processed/clean_retail.parquet


In [50]:
# 5. Print final results
print("\n--- PHASE 1 COMPLETE ---")
print(f"Original rows: {len(df):,}")
print(f"Clean rows:    {len(df_clean):,} (Retained {len(df_clean)/len(df)*100:.1f}%)")
print(f"Unique Customers: {df_clean['Customer ID'].nunique():,}")
print(f"Date range:    {df_clean['InvoiceDate'].min()} to {df_clean['InvoiceDate'].max()}")
df_clean.head()
df_clean.tail()


--- PHASE 1 COMPLETE ---
Original rows: 1,067,371
Clean rows:    805,549 (Retained 75.5%)
Unique Customers: 5,878
Date range:    2009-12-01 07:45:00 to 2011-12-09 12:50:00


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France,14.85
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680,France,18.00
